<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

# CP101: Assignment 4 - Privately Owned, Publicly Promised? (NYC POPS)

**Due Date:** Nov 4, 2026 - 11:59 pm

**Total Points:** 100 (+ 5 bonus)

---

### The story

NYC's zoning code lets developers build extra floor area in exchange for providing a **Privately Owned Public Space (POPS)** -- a plaza, arcade, or covered space that must stay open to the public. There are about 392 of them citywide. This assignment asks the same questions the Week 7 GeoPandas class activity did: which buildings are actually POPS-associated, are they taller than other buildings, and are they easy to reach by subway -- plus two new questions: does the map you draw depend on how you classify the data, and could you build an interactive plot to explore it yourself?

### Instructions
- Complete all questions in this notebook
- Use the exact variable and function names requested
- Complete all code cells, then run **Kernel -> Restart & Run All** before submitting
- Submit the autograder zip in Gradescope

In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("CP101_A4.ipynb")

In [ ]:
# Run this cell first -- it imports every package used in this notebook.
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import mapclassify
import plotly.express as px

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Part 1: Debugging & Functions (18 points)

### Q1.1 -- Semantic Bug

A planner writes a function to flag buildings taller than a threshold for a zoning review. The function runs without error, but flags the wrong buildings -- fix it so the output is `['Tower A', 'Tower C', 'Tower E']`.

In [ ]:
# BUGGY CODE
building_data = {
    'Tower A': {'heightroof': 210.0, 'groundelev': 12.0},
    'Tower B': {'heightroof': 165.0, 'groundelev': 45.0},
    'Tower C': {'heightroof': 305.0, 'groundelev': 8.0},
    'Tower D': {'heightroof': 190.0, 'groundelev': 60.0},
    'Tower E': {'heightroof': 220.0, 'groundelev': 5.0},
}

def flag_tall_buildings(buildings, threshold=200):
    flagged = []
    for name, info in buildings.items():
        if info['groundelev'] > threshold:
            flagged.append(name)
    return flagged

print(flag_tall_buildings(building_data))

In [ ]:
def flag_tall_buildings(buildings, threshold=200):
    ...

### Reading a Traceback (not graded)

The real building dataset you'll load in Part 2 records `heightroof` as **text**, not a number -- a common data-quality issue. The code below raises an error:

```python
heights = {'100 Maiden Lane': '217.5', '55 Water Street': '187.3'}

def over_threshold(heights, threshold=200):
    return [name for name, h in heights.items() if h > threshold]

over_threshold(heights)
```

**Check your understanding:** what error does this raise, and why does comparing `h` to `threshold` fail?

<details>
<summary>Reveal answer</summary>

**`TypeError`** -- `h` is a `str` (e.g. `'217.5'`) and `threshold` is an `int`. Python won't compare a string to a number with `>`, so the comparison raises instead of silently doing the wrong thing. Fix it with `pd.to_numeric(...)` (or `float(h)`) before comparing -- exactly what you'll need to do with the real `heightroof` column in Part 2.

</details>

### Q2: Counting Public Space Types

A site surveyor records the `public_space_type` of each POPS visited. Create a variable `space_type_counts` that references a dictionary mapping each type to how many times it appears in `visited_types`.

In [ ]:
visited_types = ['Plaza', 'Arcade', 'Plaza', 'Residential Plaza', 'Plaza', 'Urban Plaza', 'Arcade', 'Plaza']

In [ ]:
space_type_counts = ...

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Part 2: Spatial Join & Nearest Join (38 points + 5 bonus)

We now work with the real data: Lower Manhattan building footprints (`bldgfp_lower_manh.parquet`) and NYC's Privately Owned Public Spaces (`pops_nyc.parquet`).

### Q3: Which buildings are POPS-associated?

Load both files into `bldgs` and `pops`. Convert `bldgs['heightroof']` to a numeric type (it arrives as text). Then use a spatial join with the `intersects` predicate to find every building that intersects a POPS, keeping only the columns `['base_bbl', 'heightroof', 'cnstrct_yr', 'geometry']`. Point `bldgs_intersect_pops` at the result.

In [ ]:
bldgs = ...
pops = ...
bldgs_intersect_pops = ...

In [ ]:
grader.check("q3")

### Q4: How close is the nearest subway entrance?

Load `mta_subway_entrances.parquet` into `mta`. Load `box_lower_manh.parquet` into `box` and set its CRS to EPSG:4326. Then filter `mta` down to the entrances inside the box with `mta.within(...)`, passing the box's polygon (`box.geometry.iloc[0]`). Point `entrances_lower_manh` at the result.

Distance needs a projected CRS. Reproject both `entrances_lower_manh` and `bldgs_intersect_pops` to **EPSG:2263** (feet), then use `sjoin_nearest` (from entrances to buildings, `how='inner'`, `max_distance=100`, `distance_col='distance_ft'`) to find every POPS-associated building within 100 feet of a subway entrance. Point `entrances_2263` and `bldgs_pops_2263` at the reprojected layers, and `nearest_entrances` at the result.

In [ ]:
mta = ...
box = ...
entrances_lower_manh = ...

entrances_2263 = ...
bldgs_pops_2263 = ...

nearest_entrances = ...


In [ ]:
grader.check("q4")

### Bonus: How far do "transit desert" buildings really have to walk? (5 points)

Q4's `max_distance=100` cutoff means any POPS-associated building with **no** subway entrance within 100ft is silently dropped from `nearest_entrances` entirely -- it's not that they have no nearest entrance, just that `sjoin_nearest` was told not to report one.

Find these "transit desert" buildings: the `base_bbl` values from `bldgs_intersect_pops` that do **not** appear in `nearest_entrances`. Then run `sjoin_nearest` again on just those buildings (`how='left'`, `distance_col='distance_ft'`, **no** `max_distance` this time) against `entrances_2263` to find each one's true nearest subway entrance, however far away it is. Point `transit_desert_nearest` at the result.

In [ ]:
transit_desert_nearest = ...

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Part 3: Thematic Mapping & Classification Schemes (18 points)

### Q5.1: Two ways to classify the same data

A choropleth or classed map needs to sort continuous values (like `heightroof`) into a small number of color bins -- how you choose those bins changes what the map shows. Using `mapclassify`, classify the numeric `heightroof` column of `bldgs` (drop missing values first) into `k=5` bins two ways:

- `equal_interval_counts`: the bin counts from `mapclassify.EqualInterval(heights, k=5)`
- `quantile_counts`: the bin counts from `mapclassify.Quantiles(heights, k=5)`

Both should be lists (or arrays) of 5 counts, in bin order.

In [ ]:
# the heightroof column of bldgs with missing values dropped
heights = ...

equal_interval_counts = ...
quantile_counts = ...

In [ ]:
grader.check("q5_1")

### Q5.2: Which scheme would you use?

Look at your two count lists from Q5.1. Which scheme would you choose to shade a map of building heights in Lower Manhattan, and why?

- **A.** Equal Interval, because every bin covers the same range of heights.
- **B.** Quantiles, because a few outliers push almost all buildings into Equal Interval's lowest bin, making the map look nearly uniform.
- **C.** Equal Interval, because each of its five bins contains a similar number of buildings, so all five colors appear across the map.
- **D.** Quantiles, because Equal Interval would leave some bins empty, and an empty bin cannot be drawn on a map.

Set `q5_2_answer` to the letter of your answer, as a string (for example `"C"`).

In [ ]:
q5_2_answer = "..."

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Part 4: Interactive Visualization with Plotly Express (9 points)

### Q6.1: An interactive height-vs-transit scatter

`nearest_entrances` already carries everything you need: `heightroof` and `cnstrct_yr` came
from the building side of the join, and `distance_ft` came from the join itself.

Build an interactive scatter plot with `plotly.express`, and point a variable named `fig` at it:

- `distance_ft` on the x-axis, `heightroof` on the y-axis
- **color the points by `cnstrct_yr`**, so the plot carries a third variable
- `base_bbl` in the hover data, so any point can be traced back to a real building
- a clear title, and axis labels that state their units

One catch, and it is the same one you hit in Q3: `cnstrct_yr` arrives as **text**. Plotly will
treat 64 year-strings as 64 unrelated categories and hand you an unreadable legend. Convert it
with `pd.to_numeric(..., errors="coerce")` first so it becomes a continuous color scale.

Finish with `fig.show()`.

In [ ]:
# cnstrct_yr is stored as text -- make it numeric first
nearest_entrances["cnstrct_yr"] = ...

fig = ...
fig.show()

### Q6.2: Read your own plot

In 2-3 sentences: does the plot suggest any relationship between a building's height and its
distance to the nearest subway entrance, or does it look like noise? Does coloring by
`cnstrct_yr` reveal anything the two-variable version would have hidden?

Before you answer, look again at where these 64 points came from. Q4 filtered with
`max_distance=100`, so every point here is a building that *already* had an entrance within
100 feet. Say what that does to the range of `distance_ft` you are looking at, and what it
means for any trend you think you see.

*Type your answer here*

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Part 5: Visualization Critique (17 points)

The chart below claims to show building heights by borough.

In [ ]:
# BUGGY CHART
boroughs = ['Manhattan', 'Brooklyn', 'Queens', 'Bronx', 'Staten Island']
avg_height = [414, 68, 52, 45, 31]

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(boroughs, avg_height, color=['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6'])
ax.set_ylim(25, 420)
ax.set_title('POPS Buildings Are MUCH Taller in Manhattan!')
plt.show()

### Q7.1 -- Identify the Issue

Exactly **one** of the following deceptive-chart categories applies to the chart above. Create a variable `q7_1_issue` that references the exact keyword string for the one that applies:

- `"truncated_axis"` -- starting the Y-axis at a value other than zero, which exaggerates differences between bars or data points.
- `"inverted_axis"` -- reversing the direction of an axis (e.g. an upward trend appearing downward), causing message reversal.
- `"dual_axis"` -- using two different Y-axes on one chart, which can mislead by letting unrelated scales create false correlations or mask trends.
- `"aspect_ratio_distortion"` -- scaling a chart's width-to-height ratio to exaggerate or minimize the perceived rate of change in a line chart.
- `"distorted_projection"` -- using 3D effects or perspective in a pie/bar chart to make some segments appear larger than they are.
- `"data_visual_disproportion"` -- mapping data to the *radius* of a bubble or circle instead of its area, which visually inflates differences.
- `"inappropriate_continuous_encoding"` -- using a line chart for categorical data, which falsely implies continuity or a trend.
- `"inappropriate_categorical_encoding"` -- using a bar chart for continuous data, which disrupts the perception of gradual variation.

Example: `q7_1_issue = "dual_axis"`

In [ ]:
q7_1_issue = ...

### Q7.2 -- Explain

Explain why this chart is misleading. Your answer should reference the **proportional ink** principle: in a bar chart, the *area* of each bar is what visually encodes its value, so that area must stay proportional to the value -- why does that requirement not apply the same way to a line chart?

*Type your answer here*

### Q7.3 -- Fix

Recreate the chart honestly, and point a variable named `ax` at the resulting `Axes` object.

Three things have to change, and only the first one is the axis:

1. **Start the y-axis at zero**, so bar area is proportional to value again.
2. **Label the y-axis**, including units. The original never said whether 414 was feet,
   meters, or storeys.
3. **Rewrite the title.** `POPS Buildings Are MUCH Taller in Manhattan!` hands the reader a
   conclusion instead of letting them read one. A chart that has to shout is doing the
   deceiving, not just the axis.

While you are there: the original used five different colors for five bars showing a single
measure. Color that encodes nothing invites the reader to look for meaning that is not there.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(boroughs, avg_height, color="#003262")
ax.set_ylim(...)
ax.set_ylabel(...)
ax.set_title(...)
plt.show()

## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a zip file for you to submit.

Make sure you have saved your notebook before running this cell. Then run this cell and click on the download link to get your submission zip.

In [ ]:
grader.export(pdf=False, force_save=True)